# 🔬 TraceScope AI 2.0 — Google Colab Pro Master Execution Pipeline
### High-Performance Deep Learning & Forensic Model Training (A100 / L4 GPU)

> **Repository:** [SWAPNILSHAW/TraceScope](https://github.com/SWAPNILSHAW/TraceScope)  
> **Target System:** TraceScope AI 2.0 (Scanner Source Attribution + Document Forensics)  
> **Current Phase:** Phase 5 (ResNet-18 Deep Baseline) & Phase 6 (Hybrid CNN)  

---

## ⚙️ Step 0: Check GPU & Hardware Acceleration

In [ ]:
!nvidia-smi
import torch
import tensorflow as tf
print("PyTorch CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
print("TensorFlow GPUs:", tf.config.list_physical_devices('GPU'))

## 📁 Step 1: Mount Google Drive & Clone/Pull Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
%cd /content
if not os.path.exists('TraceScope'):
    !git clone https://github.com/SWAPNILSHAW/TraceScope.git
%cd /content/TraceScope
!git pull origin main

## 📦 Step 2: Install Required Dependencies

In [ ]:
!pip install -q scikit-image tifffile PyWavelets seaborn gdown

## 💾 Step 3: Link/Verify Pre-computed Residual Cache (1.19 GB)

In [ ]:
import os
import shutil

target_dir = '/content/TraceScope/results/hybrid_cnn'
os.makedirs(target_dir, exist_ok=True)
target_file = os.path.join(target_dir, 'official_wiki_residuals.pkl')

# Check if present in Google Drive, copy if found
drive_search_paths = [
    '/content/drive/MyDrive/official_wiki_residuals.pkl',
    '/content/drive/MyDrive/TraceScope/results/hybrid_cnn/official_wiki_residuals.pkl',
    '/content/drive/MyDrive/results/hybrid_cnn/official_wiki_residuals.pkl'
]

found = False
if os.path.exists(target_file):
    print(" Residual cache already present at target location.")
    found = True
else:
    for p in drive_search_paths:
        if os.path.exists(p):
            print(f" Found in Google Drive: {p}")
            shutil.copy(p, target_file)
            found = True
            break

# If not found in drive, download model folder from Readme Google Drive link
if not found:
    print(" Downloading residual cache from Google Drive link...")
    !gdown --folder "https://drive.google.com/drive/folders/1RdbKpvlMvLe7t77KOmmgz8ugZy4vTQeT" -O /content/TraceScope/results/hybrid_cnn/

print("Residual cache status:", "READY ✅" if os.path.exists(target_file) else "NOT FOUND ❌")

## 🚀 Step 4: Phase 5 — Train Deep CNN Baseline (ResNet-18 on GPU)
- Backbone: ResNet-18 initialized from scratch (no natural image bias)
- Filter: Fixed $5 \times 5$ Kraetzer-Vogler High-Pass Kernel
- Splits: `splits/train_manifest.csv` $\to$ Evaluated on locked `splits/test_manifest.csv`
- Outputs: `models/cnn/resnet18_best.pth`, `results/cnn/training_history.csv`, `test_metrics.csv`

In [ ]:
# Train PyTorch ResNet-18 with Kraetzer-Vogler High-Pass Filter on Colab A100 GPU
!python src/cnn_model/train.py --epochs 25 --batch_size 64 --lr 0.0005

## 🚀 Step 5: Phase 6 — Train Hybrid CNN Primary Model (Dual-Branch Fusion)
- Branch A: $256 \times 256$ Residual CNN (Conv2D 32-64-128 + GAP)
- Branch B: 44 Handcrafted Forensic Features (PRNU + FFT + LBP + Texture)
- Outputs: `models/hybrid_cnn/scanner_hybrid.keras`, training plots, confusion matrix

In [ ]:
# Train TensorFlow/Keras Hybrid CNN on Colab GPU
!python src/hybrid_cnn/train_hybrid_cnn.py

## 💾 Step 6: Backup Trained Checkpoints & Metrics to Google Drive

In [ ]:
import os
backup_dir = '/content/drive/MyDrive/TraceScope_2.0_Phase5_Phase6_Backup'
os.makedirs(backup_dir, exist_ok=True)

# Backup models and results
!cp -r models /content/drive/MyDrive/TraceScope_2.0_Phase5_Phase6_Backup/
!cp -r results /content/drive/MyDrive/TraceScope_2.0_Phase5_Phase6_Backup/
print("✅ All models, learning curves, and test metrics successfully backed up to Google Drive!")